<a href="https://colab.research.google.com/github/wlgns222/ROKA/blob/main/ai-study/deep-learning-with-pytorch/3_Tensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3. It starts with Tensor

## 3.2 Tensors: Multidimensional arrays

### 3.2.1 From Python lists to Pytorch Tensors

In [1]:
a = [1.0, 2.0, 1.0]

In [2]:
a[0]

1.0

In [3]:
a[2] = 3.0
a

[1.0, 2.0, 3.0]

파이썬 프로그램에서는 2차원에서 선의 좌표를 숫자 벡터를 다루기 위해 파이썬 리스트를 사용하는 일이 흔하다.

텐서에 대한 연산을 정의해두면 동일한 작업에 대해 파이썬 같은 고차원 언어보다 훨씬 효율적이고 알아보기 쉽게 데이터를 자르고 조작할 수 있다.

### 3.2.2 Constructing our first Tensor

In [4]:
import torch

b = torch.ones(3)
b

tensor([1., 1., 1.])

1.0 으로 이루어진 3의 크기를 가진 일차원 텐서를 만든다.

In [6]:
print(b[1])
print(float(b[1]))
b[2] = 2.0
print(b)

tensor(1.)
1.0
tensor([1., 1., 2.])


### 3.2.3 The Essence of Tensors

숫자값으로 만든 파이썬 리스트나 튜플 객체는 메모리에 따로따로 할당된다. 반면 파이토치 텐서나 넘파이 배열은 파이썬 객체가 아닌 **언박싱** 된 C 언어 숫자 타입을 연속적으로 메모리에 할당되고 이에 대한 뷰를 제공한다.

각 요소는 32비트(4바이트) float 타입이며 100만개의 float 타입 숫자를 1차원 텐서에 보관한다면 400만 바이트의 연속적인 공간과 (차원이나 숫자 타입을 기록하는 용도의) 메타데이터 공간을 조금 더 차지한다.

## 3.3 Indexing Tensors

In [12]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])

# 1. 특정 원소 하나만 가져오기
print(points[0, 1])
# 2. 특정 행 전체 가져오기
print(points[0])

tensor(1.)
tensor([4., 1.])


In [15]:
# [행 전체, 0번 열]
print(points[:, 0])
# 첫 번째 행 이후의 모든 데이터
print(points[1:])
#None 을 이용해 차원 늘리기
print(points[None, 1:3])

tensor([4., 5., 2.])
tensor([[5., 3.],
        [2., 1.]])
tensor([[[5., 3.],
         [2., 1.]]])


## 3.4 Named Tensors

텐서는 차원이나 축이 있으며, 각 차원은 픽셀 위치나 컬러 채널에 해당한다.

때문에 텐서를 접근하려면 차원의 순서를 기억해서 인덱싱해야 한다. 이때, 데이터가 여러 텐서 형태를 거치며 다양하게 변환되면, 어느 차원에 어느 데이터가 들어있는지 헷갈려 실수하기 쉽다.

이미지 데이터를 흑백으로 변환한다고 가정해보자.

In [19]:
img_t = torch.randn(3, 5, 5) # [채널크기, 행크기, 열크기]
batch_t = torch.randn(2, 3, 5, 5)
weights = torch.tensor([0.2126, 0.7152, 0.0722])

In [22]:
img_gray_native = img_t.mean(-3)
batch_gray_native = batch_t.mean(-3)
img_gray_native.shape, batch_gray_native.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

파이토치는 동일한 차원 정보의 텐서끼리 연산할 수 있고, 각 길이가 1인 텐서도 가능하다. 혹은 길이가 1인 차원을 알아서 늘려주기도 하는데, 이를 **브로드캐스팅** 이라고 한다.


> Numpy ndarray 와 동일한 수학적 규칙을 사용하나 파이토치에서는 브로드캐스팅된 연산을 GPU 의 수천 개의 코어에서 동시에 처리한다.



In [23]:
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze(-1)
img_weights = (img_t * unsqueezed_weights)
batch_weights = (batch_t * unsqueezed_weights)

img_gray_weighted = img_weights.sum(-3)
batch_gray_weighted = batch_weights.sum(-3)

batch_weights.shape, batch_t.shape, unsqueezed_weights.shape

(torch.Size([2, 3, 5, 5]), torch.Size([2, 3, 5, 5]), torch.Size([3, 1, 1]))

**unsqueeze 의 목적** : 브로드캐스팅 준비

- 원본 가중치의 형태는 [3]이나, 이미지는 [3, H, W] 형태
- weights.unsqueeze(-1) -> [3, 1] -> .unsqueeze(-1) -> [3, 1, 1]
- 모양이 [3, 1, 1] 이 되어 이미지와 계산할 수 있는 상태 (브로드캐스팅 가능)

**sum(-3) 의 목적** : 채널 통합(흑백화)

가중치를 곱한 후 우리는 [3, H, W] 형태의 결과물을 얻는다. 흑백 이미지는 채널이 1개여야 하므로, 이 3개의 채널을 하나로 합쳐야한다.

파이토치는 각 차원에 이름을 붙여 관리할 수 있게한다.

In [20]:
img_named = img_t.refine_names('channels', 'rows', 'columns')

batch_named = batch_t.refine_names(..., 'channels', 'rows', 'columns')

weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names=['channels'])

print(weights_named)
print("img_named :", img_named.shape, img_named.names)
print("batch_named :", batch_named.shape, batch_named.names)

tensor([0.2126, 0.7152, 0.0722], names=('channels',))
img_named : torch.Size([3, 5, 5]) ('channels', 'rows', 'columns')
batch_named : torch.Size([2, 3, 5, 5]) (None, 'channels', 'rows', 'columns')


## 3.5 Tensor element types

Tensor 에 어떤 타입의 값을 저장할 수 있는지에 대해 다룬다.

표준 파이썬 숫자 타입은 여러 이유에서 최적이 아니다.
- **파이썬에서 숫자는 객체이다**
    
    통상 부동소수점 수는 컴퓨터에서 32비트의 공간을 차지한다. 그러나 파이썬은 부동소수점 수를 완전한 파이썬 객체로 변환한다. **박싱**으로 부르는 이 연산은 수가 백만 개가 넘어가면 상당히 비효율적이다.
- **파이썬에서 리스트는 연속된 객체의 컬렉션이다**
- **파이썬 인터프리터는 최적화를 거치는 컴파일된 코드보다 느리다**

### 3.5.1 Specifying the numeric type with dtype

### 3.5.2 A dtype for every occasion

### 3.5.3 Managing a tensor’s dtype attribute

## 3.6 The Tensors API

## 3.7 Tensors: Scenic views of storage

### 3.7.1 Indexing into storage

### 3.7.2 Modifying stored values: In-place operations

## 3.8 Tensor metadata: Size, offset, and stride

### 3.8.1 Views of another tensor’s storage

### 3.8.2 Transposing without copying

### 3.8.3 Transposing in higher dimensions

### 3.8.4 Contiguous tensors

## 3.9 Moving tensors to the GPU

### 3.9.1 Managing a tensor’s device attribute

## 3.10 NumPy interoperability

## 3.11 Generalized tensors are tensors, too